In [1]:
# ============================================================
# CIC-PQC_OAV v1
# MACHINE LEARNING-BASED ANOMALY DETECTION IN
# POST-QUANTUM TLS 1.3 TRAFFIC
#
# DATA PREPROCESSING ACCORDING TO:
# Keskin, S. (2026)
# "Machine Learning-Based Anomaly Detection in Post-Quantum
# TLS 1.3 Traffic: A Comparative Analysis on the
# CIC-PQC_OAV v1 Dataset"
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer


# ============================================================
# 2. LOAD DATASET
# ============================================================

FILE_PATH = "ml_features_and_labels.csv"

df = pd.read_csv(FILE_PATH)

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)

print(f"Number of rows    : {df.shape[0]}")
print(f"Number of columns : {df.shape[1]}")

print("\nFirst 5 rows:")
display(df.head())

print("\nColumn names:")
for i, col in enumerate(df.columns):
    print(f"{i+1:2d}. {col}")


# ============================================================
# 3. BASIC DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

print(df.info())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))


# ============================================================
# 4. CHECK FOR DUPLICATE RECORDS
# ============================================================

print("\n" + "=" * 70)
print("DUPLICATE CHECK")
print("=" * 70)

duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")

if duplicate_count > 0:
    print("WARNING: Duplicate rows were found.")
else:
    print("No duplicate rows found.")


# ============================================================
# 5. CHECK REQUIRED COLUMNS
# ============================================================

print("\n" + "=" * 70)
print("REQUIRED COLUMN CHECK")
print("=" * 70)

required_columns = [
    "ID",
    "split",
    "taxonomy",
    "label"
]

for col in required_columns:
    if col in df.columns:
        print(f"[OK] {col}")
    else:
        print(f"[MISSING] {col}")

missing_required = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required:
    raise ValueError(
        f"Required columns missing from dataset: {missing_required}"
    )


# ============================================================
# 6. TARGET VARIABLE ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("TARGET VARIABLE: label")
print("=" * 70)

print("\nUnique label values:")
print(df["label"].unique())

print("\nLabel distribution:")
label_counts = df["label"].value_counts().sort_index()

display(
    pd.DataFrame({
        "count": label_counts,
        "percentage": (label_counts / len(df) * 100).round(2)
    })
)

print("\nExpected:")
print("0 = Normal")
print("1 = Anomaly")


# Convert label to numeric if necessary
df["label"] = pd.to_numeric(df["label"], errors="coerce")

if df["label"].isna().any():
    raise ValueError("Some label values could not be converted to numeric.")

df["label"] = df["label"].astype(int)


# ============================================================
# 7. CHECK THE PREDEFINED SPLIT
# ============================================================

print("\n" + "=" * 70)
print("PREDEFINED DATASET SPLIT")
print("=" * 70)

print("Unique split values:")
print(df["split"].unique())

print("\nSplit distribution:")
split_counts = df["split"].value_counts()

display(
    pd.DataFrame({
        "count": split_counts,
        "percentage": (split_counts / len(df) * 100).round(2)
    })
)


# ============================================================
# 8. ANALYZE LABEL DISTRIBUTION WITHIN EACH SPLIT
# ============================================================

print("\n" + "=" * 70)
print("LABEL DISTRIBUTION BY SPLIT")
print("=" * 70)

split_label_table = pd.crosstab(
    df["split"],
    df["label"]
)

print(split_label_table)

print("\nPercentage distribution:")
split_label_percentage = pd.crosstab(
    df["split"],
    df["label"],
    normalize="index"
) * 100

display(split_label_percentage.round(2))


# ============================================================
# 9. IDENTIFY COLUMNS TO EXCLUDE
# ============================================================

print("\n" + "=" * 70)
print("FEATURE SELECTION")
print("=" * 70)

TARGET_COLUMN = "label"

EXCLUDED_COLUMNS = [
    "ID",
    "split",
    "taxonomy"
]

print("Target column:")
print(TARGET_COLUMN)

print("\nExcluded columns:")
for col in EXCLUDED_COLUMNS:
    print(f"- {col}")


# ============================================================
# 10. CREATE FEATURE MATRIX
# ============================================================

feature_columns = [
    col for col in df.columns
    if col not in EXCLUDED_COLUMNS + [TARGET_COLUMN]
]

print("\nNumber of independent features:", len(feature_columns))

print("\nIndependent features:")
for i, col in enumerate(feature_columns):
    print(f"{i+1:2d}. {col}")


# According to the paper:
# 36 total columns
# - ID
# - split
# - taxonomy
# - label
# = 32 independent features

if len(feature_columns) != 32:
    print(
        f"\nWARNING: The paper expects 32 independent features, "
        f"but this CSV contains {len(feature_columns)}."
    )


# ============================================================
# 11. CREATE X AND y
# ============================================================

X = df[feature_columns].copy()
y = df[TARGET_COLUMN].copy()

print("\nFeature matrix shape:", X.shape)
print("Target shape        :", y.shape)


# ============================================================
# 12. CHECK FEATURE DATA TYPES
# ============================================================

print("\n" + "=" * 70)
print("FEATURE DATA TYPES")
print("=" * 70)

dtype_table = pd.DataFrame({
    "feature": X.columns,
    "dtype": X.dtypes.astype(str).values
})

display(dtype_table)


# ============================================================
# 13. CONVERT BOOLEAN FEATURES TO 0/1
# ============================================================

print("\n" + "=" * 70)
print("BOOLEAN / LOGICAL FEATURE CONVERSION")
print("=" * 70)

boolean_columns = X.select_dtypes(
    include=["bool"]
).columns.tolist()

print("Boolean columns detected:")
print(boolean_columns)

for col in boolean_columns:
    X[col] = X[col].astype(int)

print("\nBoolean conversion completed.")


# ============================================================
# 14. HANDLE OBJECT / STRING FEATURES
# ============================================================

# The ML-ready dataset should contain numerical/binary features.
# This section detects unexpected string columns.

object_columns = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("\nObject/category columns detected:")
print(object_columns)

if len(object_columns) > 0:
    print(
        "\nWARNING: These columns contain non-numeric values."
    )
    print(
        "They need to be inspected before scaling."
    )

    for col in object_columns:
        print(f"\n{col}:")
        print(X[col].value_counts().head(10))


# ============================================================
# 15. CONVERT NUMERIC-LIKE FEATURES TO NUMERIC
# ============================================================

for col in X.columns:

    if X[col].dtype == "object":

        converted = pd.to_numeric(
            X[col],
            errors="coerce"
        )

        # If conversion produces fewer missing values,
        # use the converted numeric representation.
        if converted.notna().sum() >= X[col].notna().sum():

            X[col] = converted


# Check again
object_columns = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

if object_columns:
    raise ValueError(
        f"Non-numeric feature columns remain: {object_columns}"
    )


# ============================================================
# 16. MISSING VALUE ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUE ANALYSIS")
print("=" * 70)

missing_counts = X.isnull().sum()

missing_table = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percentage": (
        missing_counts / len(X) * 100
    ).round(3)
})

missing_table = missing_table[
    missing_table["missing_count"] > 0
].sort_values(
    "missing_count",
    ascending=False
)

if len(missing_table) > 0:
    display(missing_table)
else:
    print("No missing values found.")


# According to the paper, missing values were reported in:
#
# e5_entropy_c
# e6_time_char
# e2_client_size
# e2_server_record_len


# ============================================================
# 17. SEPARATE TRAIN / VALIDATION / TEST USING
#     THE ORIGINAL "split" COLUMN
# ============================================================

print("\n" + "=" * 70)
print("CREATING TRAIN / VALIDATION / TEST SETS")
print("=" * 70)


# Display exact split values first
print("Available split values:")
print(df["split"].unique())


# Automatically detect common names
split_values = df["split"].astype(str).str.lower()

train_mask = split_values.isin([
    "train",
    "training"
])

val_mask = split_values.isin([
    "val",
    "validation",
    "valid"
])

test_mask = split_values.isin([
    "test",
    "testing"
])


# ------------------------------------------------------------
# If the dataset uses another naming convention, print it
# rather than silently making an incorrect split.
# ------------------------------------------------------------

if train_mask.sum() == 0:
    raise ValueError(
        "Could not identify the training partition from the "
        "'split' column."
    )

if val_mask.sum() == 0:
    raise ValueError(
        "Could not identify the validation partition from the "
        "'split' column."
    )

if test_mask.sum() == 0:
    raise ValueError(
        "Could not identify the test partition from the "
        "'split' column."
    )


# Create partitions
X_train = X.loc[train_mask].copy()
X_val   = X.loc[val_mask].copy()
X_test  = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_val   = y.loc[val_mask].copy()
y_test  = y.loc[test_mask].copy()


print("\nPartition sizes:")
print(f"Training   : {X_train.shape}")
print(f"Validation : {X_val.shape}")
print(f"Test       : {X_test.shape}")


# ============================================================
# 18. VERIFY EXPECTED PARTITION SIZES
# ============================================================

print("\n" + "=" * 70)
print("PARTITION SIZE VERIFICATION")
print("=" * 70)

expected_train = 24006
expected_val = 8002
expected_test = 8002

print(
    f"Training   : {len(X_train)} "
    f"(expected {expected_train})"
)

print(
    f"Validation : {len(X_val)} "
    f"(expected {expected_val})"
)

print(
    f"Test       : {len(X_test)} "
    f"(expected {expected_test})"
)


# ============================================================
# 19. VERIFY EXPECTED CLASS DISTRIBUTION
# ============================================================

print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

print("\nTRAINING SET")
print(y_train.value_counts().sort_index())

print("\nVALIDATION SET")
print(y_val.value_counts().sort_index())

print("\nTEST SET")
print(y_test.value_counts().sort_index())


# ============================================================
# 20. MEDIAN IMPUTATION
# ============================================================
#
# IMPORTANT:
# The paper explicitly states that median values are calculated
# using TRAINING DATA ONLY.
#
# Then the same training-derived medians are applied to
# validation and test data.
#
# This prevents data leakage.
# ============================================================

print("\n" + "=" * 70)
print("MEDIAN IMPUTATION")
print("=" * 70)

imputer = SimpleImputer(
    strategy="median"
)

# FIT ONLY ON TRAINING DATA
X_train_imputed = imputer.fit_transform(X_train)

# TRANSFORM validation and test using training medians
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)


# Convert back to DataFrames
X_train = pd.DataFrame(
    X_train_imputed,
    columns=feature_columns,
    index=X_train.index
)

X_val = pd.DataFrame(
    X_val_imputed,
    columns=feature_columns,
    index=X_val.index
)

X_test = pd.DataFrame(
    X_test_imputed,
    columns=feature_columns,
    index=X_test.index
)


# Show medians learned from training data
training_medians = pd.Series(
    imputer.statistics_,
    index=feature_columns
)

print("\nTraining-set medians used for imputation:")

median_table = pd.DataFrame({
    "feature": feature_columns,
    "training_median": training_medians.values
})

display(median_table)


# ============================================================
# 21. VERIFY NO MISSING VALUES REMAIN
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUE CHECK AFTER IMPUTATION")
print("=" * 70)

print(
    "Training missing values   :",
    X_train.isnull().sum().sum()
)

print(
    "Validation missing values :",
    X_val.isnull().sum().sum()
)

print(
    "Test missing values       :",
    X_test.isnull().sum().sum()
)


# ============================================================
# 22. IDENTIFY NUMERICAL FEATURES
# ============================================================
#
# The ML-ready CIC-PQC_OAV features are numerical/binary.
#
# We identify columns with more than two unique numerical values
# as continuous/numerical variables.
#
# Binary columns (0/1) are retained as binary values.
# ============================================================

numerical_columns = []

binary_columns = []

for col in feature_columns:

    unique_values = X_train[col].dropna().unique()

    if len(unique_values) <= 2:

        binary_columns.append(col)

    else:

        numerical_columns.append(col)


print("\n" + "=" * 70)
print("FEATURE TYPES")
print("=" * 70)

print(
    f"Numerical/continuous features : {len(numerical_columns)}"
)

print(
    f"Binary features               : {len(binary_columns)}"
)

print("\nNumerical features:")
for col in numerical_columns:
    print("-", col)

print("\nBinary features:")
for col in binary_columns:
    print("-", col)


# ============================================================
# 23. STANDARDIZATION
# ============================================================
#
# StandardScaler:
#
# z = (x - mean) / standard deviation
#
# IMPORTANT:
# The scaler is FIT ONLY on training data.
# Validation and test data are transformed using the same
# training-derived parameters.
# ============================================================

print("\n" + "=" * 70)
print("STANDARDIZATION")
print("=" * 70)

scaler = StandardScaler()

# Fit ONLY on training numerical features
X_train[numerical_columns] = scaler.fit_transform(
    X_train[numerical_columns]
)

# Apply same transformation to validation
X_val[numerical_columns] = scaler.transform(
    X_val[numerical_columns]
)

# Apply same transformation to test
X_test[numerical_columns] = scaler.transform(
    X_test[numerical_columns]
)


print("StandardScaler fitted on training data.")
print("Validation and test transformed using training parameters.")


# ============================================================
# 24. VERIFY STANDARDIZATION
# ============================================================

print("\n" + "=" * 70)
print("STANDARDIZATION CHECK")
print("=" * 70)

scaling_check = pd.DataFrame({
    "mean": X_train[numerical_columns].mean(),
    "std": X_train[numerical_columns].std()
})

display(scaling_check.round(4))


# ============================================================
# 25. VERIFY BINARY FEATURES
# ============================================================

print("\n" + "=" * 70)
print("BINARY FEATURE CHECK")
print("=" * 70)

for col in binary_columns:

    unique_values = sorted(
        X_train[col].dropna().unique().tolist()
    )

    print(f"{col}: {unique_values}")


# ============================================================
# 26. FINAL DATASET SHAPES
# ============================================================

print("\n" + "=" * 70)
print("FINAL PREPROCESSED DATA")
print("=" * 70)

print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape}")

print(f"X_val   : {X_val.shape}")
print(f"y_val   : {y_val.shape}")

print(f"X_test  : {X_test.shape}")
print(f"y_test  : {y_test.shape}")


# ============================================================
# 27. RESET INDICES
# ============================================================

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)


# ============================================================
# 28. FINAL SANITY CHECK
# ============================================================

print("\n" + "=" * 70)
print("FINAL SANITY CHECK")
print("=" * 70)

print(
    "Training NaNs   :",
    X_train.isnull().sum().sum()
)

print(
    "Validation NaNs :",
    X_val.isnull().sum().sum()
)

print(
    "Test NaNs       :",
    X_test.isnull().sum().sum()
)

print(
    "Training shape   :",
    X_train.shape
)

print(
    "Validation shape :",
    X_val.shape
)

print(
    "Test shape       :",
    X_test.shape
)


# ============================================================
# 29. FINAL SUMMARY TABLE
# ============================================================

summary = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation",
        "Test",
        "Total"
    ],

    "Samples": [
        len(X_train),
        len(X_val),
        len(X_test),
        len(X_train) + len(X_val) + len(X_test)
    ],

    "Features": [
        X_train.shape[1],
        X_val.shape[1],
        X_test.shape[1],
        X_train.shape[1]
    ],

    "Normal (0)": [
        (y_train == 0).sum(),
        (y_val == 0).sum(),
        (y_test == 0).sum(),
        (y == 0).sum()
    ],

    "Anomaly (1)": [
        (y_train == 1).sum(),
        (y_val == 1).sum(),
        (y_test == 1).sum(),
        (y == 1).sum()
    ]
})

display(summary)


# ============================================================
# 30. SAVE PREPROCESSED DATASETS
# ============================================================

X_train.to_csv(
    "X_train_preprocessed.csv",
    index=False
)

X_val.to_csv(
    "X_val_preprocessed.csv",
    index=False
)

X_test.to_csv(
    "X_test_preprocessed.csv",
    index=False
)

y_train.to_csv(
    "y_train.csv",
    index=False
)

y_val.to_csv(
    "y_val.csv",
    index=False
)

y_test.to_csv(
    "y_test.csv",
    index=False
)


print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("X_train_preprocessed.csv")
print("X_val_preprocessed.csv")
print("X_test_preprocessed.csv")
print("y_train.csv")
print("y_val.csv")
print("y_test.csv")

print("\nPreprocessing completed successfully.")

DATASET LOADED
Number of rows    : 40010
Number of columns : 36

First 5 rows:


,e2c_total_bytes,e4_entropy_h,e5_entropy_c,e6_time_char,e6b_flow_duration_ms,e2_client_size,e2_client_record_len,e2_server_record_len,e3_cert_parsed,e1_alg_suite_Unknown(0x11eb),...,e1b_ciphersuite_60,e2b_tls_version_SSL3.0,e2b_tls_version_TLS1.0,e2b_ciphersuite_2,e2b_ciphersuite_54,e2b_ciphersuite_60,label,split,taxonomy,ID
0,8182,4.7889,5.8077,2.45,149.827957,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,train,eval_test_network,net_high_jitter_classic_run1
1,8126,5.0000,5.8367,2.44,231.020927,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,val,eval_test_network,net_high_jitter_classic_run10
2,8272,4.7889,5.9056,2.73,292.967796,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,train,eval_test_network,net_high_jitter_classic_run100
3,8272,4.8125,5.9056,2.31,156.050205,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,test,eval_test_network,net_high_jitter_classic_run101
4,8272,4.8125,5.8367,4.45,193.972111,32.0,285,5316.0,False,False,...,True,False,True,False,False,True,1,test,eval_test_network,net_high_jitter_classic_run102



Column names:
 1. e2c_total_bytes
 2. e4_entropy_h
 3. e5_entropy_c
 4. e6_time_char
 5. e6b_flow_duration_ms
 6. e2_client_size
 7. e2_client_record_len
 8. e2_server_record_len
 9. e3_cert_parsed
10. e1_alg_suite_Unknown(0x11eb)
11. e1_alg_suite_Unknown(0x11ec)
12. e1_alg_suite_mlkem1024
13. e1_alg_suite_mlkem768
14. e1_alg_suite_x25519
15. e1_alg_suite_nan
16. e8_proto_context_TLSv1.2_or_older
17. e8_proto_context_TLSv1.3
18. e9_conn_outcome_Failure
19. e9_conn_outcome_Incomplete
20. e9_conn_outcome_Success
21. e3_sig_alg_Unknown(0x0028)
22. e3_sig_alg_nan
23. e1b_tls_version_SSL3.0
24. e1b_tls_version_TLS1.0
25. e1b_ciphersuite_2
26. e1b_ciphersuite_54
27. e1b_ciphersuite_60
28. e2b_tls_version_SSL3.0
29. e2b_tls_version_TLS1.0
30. e2b_ciphersuite_2
31. e2b_ciphersuite_54
32. e2b_ciphersuite_60
33. label
34. split
35. taxonomy
36. ID

DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40010 entries, 0 to 40009
Data columns (total 36 columns):
 #   Column        

,dtype
e2c_total_bytes,int64
e4_entropy_h,float64
e5_entropy_c,float64
e6_time_char,float64
e6b_flow_duration_ms,float64
e2_client_size,float64
e2_client_record_len,int64
e2_server_record_len,float64
e3_cert_parsed,bool
e1_alg_suite_Unknown(0x11eb),bool



DUPLICATE CHECK
Duplicate rows: 0
No duplicate rows found.

REQUIRED COLUMN CHECK
[OK] ID
[OK] split
[OK] taxonomy
[OK] label

TARGET VARIABLE: label

Unique label values:
[1 0]

Label distribution:


...


MISSING VALUE CHECK AFTER IMPUTATION
Training missing values   : 0
Validation missing values : 0
Test missing values       : 0

FEATURE TYPES
Numerical/continuous features : 8
Binary features               : 24

Numerical features:
- e2c_total_bytes
- e4_entropy_h
- e5_entropy_c
- e6_time_char
- e6b_flow_duration_ms
- e2_client_size
- e2_client_record_len
- e2_server_record_len

Binary features:
- e3_cert_parsed
- e1_alg_suite_Unknown(0x11eb)
- e1_alg_suite_Unknown(0x11ec)
- e1_alg_suite_mlkem1024
- e1_alg_suite_mlkem768
- e1_alg_suite_x25519
- e1_alg_suite_nan
- e8_proto_context_TLSv1.2_or_older
- e8_proto_context_TLSv1.3
- e9_conn_outcome_Failure
- e9_conn_outcome_Incomplete
- e9_conn_outcome_Success
- e3_sig_alg_Unknown(0x0028)
- e3_sig_alg_nan
- e1b_tls_version_SSL3.0
- e1b_tls_version_TLS1.0
- e1b_ciphersuite_2
- e1b_ciphersuite_54
- e1b_ciphersuite_60
- e2b_tls_version_SSL3.0
- e2b_tls_version_TLS1.0
- e2b_ciphersuite_2
- e2b_ciphersuite_54
- e2b_ciphersuite_60

STANDARDIZATION


,mean,std
e2c_total_bytes,0.0,1.0
e4_entropy_h,-0.0,1.0
e5_entropy_c,-0.0,1.0
e6_time_char,0.0,1.0
e6b_flow_duration_ms,0.0,1.0
e2_client_size,0.0,1.0
e2_client_record_len,-0.0,1.0
e2_server_record_len,0.0,1.0



BINARY FEATURE CHECK
e3_cert_parsed: [0.0, 1.0]
e1_alg_suite_Unknown(0x11eb): [0.0, 1.0]
e1_alg_suite_Unknown(0x11ec): [0.0, 1.0]
e1_alg_suite_mlkem1024: [0.0, 1.0]
e1_alg_suite_mlkem768: [0.0, 1.0]
e1_alg_suite_x25519: [0.0, 1.0]
e1_alg_suite_nan: [0.0, 1.0]
e8_proto_context_TLSv1.2_or_older: [0.0, 1.0]
e8_proto_context_TLSv1.3: [0.0, 1.0]
e9_conn_outcome_Failure: [0.0, 1.0]
e9_conn_outcome_Incomplete: [0.0, 1.0]
e9_conn_outcome_Success: [0.0, 1.0]
e3_sig_alg_Unknown(0x0028): [0.0, 1.0]
e3_sig_alg_nan: [0.0, 1.0]
e1b_tls_version_SSL3.0: [0.0, 1.0]
e1b_tls_version_TLS1.0: [0.0, 1.0]
e1b_ciphersuite_2: [0.0, 1.0]
e1b_ciphersuite_54: [0.0, 1.0]
e1b_ciphersuite_60: [0.0, 1.0]
e2b_tls_version_SSL3.0: [0.0, 1.0]
e2b_tls_version_TLS1.0: [0.0, 1.0]
e2b_ciphersuite_2: [0.0, 1.0]
e2b_ciphersuite_54: [0.0, 1.0]
e2b_ciphersuite_60: [0.0, 1.0]

FINAL PREPROCESSED DATA
X_train : (24006, 32)
y_train : (24006,)
X_val   : (8002, 32)
y_val   : (8002,)
X_test  : (8002, 32)
y_test  : (8002,)

FINAL SANI

,Dataset,Samples,Features,Normal (0),Anomaly (1)
0,Training,24006,32,21000,3006
1,Validation,8002,32,7000,1002
2,Test,8002,32,7000,1002
3,Total,40010,32,35000,5010



FILES SAVED
X_train_preprocessed.csv
X_val_preprocessed.csv
X_test_preprocessed.csv
y_train.csv
y_val.csv
y_test.csv

Preprocessing completed successfully.
